##### Mount Drive into Collab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##### Path to Model Repository

In [ ]:
%cd /content/drive/MyDrive/master_thesis_FM/models/1_AnySat/feature_extraction/

/content/drive/MyDrive/master_thesis_FM/models/1_AnySat/feature_extraction


##### Installing Model requirements

In [ ]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.4/99.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.2/308.2 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 157.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.2/469.2 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# [ORIGINAL PART]

# AnySat Guide

#### Simple Usage

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
import numpy as np
import rasterio
from datetime import datetime
import h5py
import time
import psutil
import matplotlib.pyplot as plt
from IPython.display import clear_output

## AnySat is available through
### 1) PyTorch Hub
#### or
### 2) local repository

In [ ]:
# 1) PyTorch Hub
#model = torch.hub.load('gastruc/anysat', 'anysat', pretrained=True, force_reload=True, flash_attn=False, trust_repo='check')

# 2) local repo
from hubconf import AnySat

model = AnySat.from_pretrained('base', flash_attn=False) #Set flash_attn=True if you have flash-attn module installed (url flash attn)
#device = "cuda" If you want to run on GPU default is cpu

Downloading: "https://huggingface.co/g-astruc/AnySat/resolve/main/models/AnySat.pth" to /root/.cache/torch/hub/checkpoints/AnySat.pth


100%|██████████| 480M/480M [00:21<00:00, 23.2MB/s]


### Added code to assure that every part of the Model runs on GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the entire model and its submodules to the GPU
model = model.to(device)

# Iterate through all model parameters and move them to the GPU if not already there
for param in model.parameters():
    if param.device != device:
        param.data = param.data.to(device)

# Check for any remaining submodules not on the GPU
for name, module in model.named_modules():
    if next(module.parameters(), None) is not None and next(module.parameters()).device != device:  # Checks if module has parameters
        module.to(device)
        print(f"Module '{name}' moved to {device}")

Module '' moved to cuda
Module 'spatial_encoder' moved to cuda
Module 'spatial_encoder.predictor_blocks' moved to cuda
Module 'spatial_encoder.predictor_blocks.0' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.norm1' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.attn' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.attn.qkv' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.attn.proj' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.attn.rpe_k' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.norm2' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.mlp' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.mlp.fc1' moved to cuda
Module 'spatial_encoder.predictor_blocks.0.mlp.fc2' moved to cuda
Module 'spatial_encoder.predictor_blocks.1' moved to cuda
Module 'spatial_encoder.predictor_blocks.1.norm1' moved to cuda
Module 'spatial_encoder.predictor_blocks.1.attn' moved to cuda
Module 'spatial_encoder.predictor_blocks.1.attn.q

# Feature Extraction

### Extraction for a complete Folder

In [ ]:
import os
import torch
import glob
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

# 🔧 Configuration
feature_set = 'test' # either 'train' or 'test'
ORBIT = 'asc'         # either 'asc' or 'desc'
OUTPUT_TYPE = 'dense' # Either 'tile', 'patch', 'dense'
CHANNELS = 3
SAMPLE_RATE = 6
BATCH_SIZE = 4
NUM_WORKERS = 8
PATCH_SIZE = 40
PRECISION_HALVING = True

# 📁 Paths
if CHANNELS == 3:
  CHANNEL_FOLDER = "3_channels_VV_VH_ratio"
elif CHANNELS == 2:
  CHANNEL_FOLDER = "2_channels_VV_VH"
else:
  raise ValueError("Invalid number of channels. Must be 2 or 3.")

if PRECISION_HALVING:
  PREC = "float16"
else:
  PREC = "float32"

INPUT_FOLDER = f"/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/s1_data/{feature_set}_partition/s1_july_pt_tensors/{CHANNEL_FOLDER}/{SAMPLE_RATE}%_subset_standardized"
OUTPUT_FOLDER = f"/content/drive/MyDrive/master_thesis_FM/datasets/BioMassters/s1_data/{feature_set}_partition/s1_july_pt_tensors/{CHANNEL_FOLDER}/Embedded_features/patch_size_{PATCH_SIZE}/{SAMPLE_RATE}%_subset_float16_TEST"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
existing_outputs = set(os.listdir(OUTPUT_FOLDER))

model_used = model  # your pretrained model should already be defined

# 📦 Dataset
class TensorDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        tensor = torch.load(path).float()  # Shape: [# channels (2 or 3), 256, 256]
        filename = os.path.basename(path)
        return tensor, filename

# 📁 Load file paths
all_paths = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.pt")))

# Filter input tensor paths to exclude already-processed files
filtered_paths = []
for path in all_paths:
    original_name = os.path.basename(path)
    expected_output_name = f"Embedded_Features_OutputType{OUTPUT_TYPE.upper()}_PatchSize{PATCH_SIZE}_{original_name}"
    if expected_output_name not in existing_outputs:
        filtered_paths.append(path)

print(f"✅ {len(filtered_paths)} of {len(all_paths)} files will be processed (skipping existing ones)")

dataset = TensorDataset(filtered_paths)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# 📤 Inference & Save
with torch.no_grad():
    for batch, filenames in tqdm(loader, desc="Extracting features"):
        batch = batch.unsqueeze(1).to(device)  # [B, 1, # channels (2 or 3), 256, 256]
        dates = torch.tensor([196] * batch.shape[0]).to(device)

        data = {
            "s1": batch,
            "s1_dates": dates
        }

        # Feature batch output from your model
        batch_features = model_used(data, patch_size=PATCH_SIZE, output=OUTPUT_TYPE, output_modality = 's1')
        print(batch_features.shape)
        print(type(batch_features))

        # 💾 Save infered encoded features
        for i in range(len(filenames)):
            original_name = filenames[i]
            output_path = os.path.join(OUTPUT_FOLDER, f"Embedded_Features_OutputType{OUTPUT_TYPE.upper()}_PatchSize{PATCH_SIZE}_{original_name}")

            # Extrahiere den Tensor für das aktuelle Bild im Batch
            single_feature = batch_features[i].cpu()
            print(single_feature.shape)
            print(single_feature.dtype)
            print(single_feature.nbytes)
            print(type(single_feature))

            # --- NEUER SHAPE-CHECK ---
            expected_dims_sorted = [256, 256, 1536]
            if sorted(list(single_feature.shape)) != expected_dims_sorted:
                raise ValueError(
                    f"Abbruch! Unerwartete Shape bei Datei {original_name}.\n"
                    f"Erwartet: Permutation von {expected_dims_sorted}\n"
                    f"Erhalten: {list(single_feature.shape)}"
                )
            # -------------------------

            # Konvertiere den Tensor explizit zu float16 vor dem Speichern
            if PRECISION_HALVING:
                single_feature = single_feature.to(torch.float16)

            torch.save(single_feature, output_path)

        # 🧹 Memory cleanup
        del batch, filenames, data, batch_features, single_feature
        torch.cuda.empty_cache()

print("✅ All encoded features extracted and saved.")

✅ 159 of 167 files will be processed (skipping existing ones)


Extracting features:   0%|          | 0/40 [00:00<?, ?it/s]

torch.Size([4, 256, 256, 1536])
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([4, 256, 256, 1536])
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([4, 256, 256, 1536])
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>
torch.Size([256, 256, 1536])
torch.float32
402653184
<class 'torch.Tensor'>